# Explore the platform

A tour of what the lake holds and of the guarantees that the code enforces.

Every read here uses `sdp.dal`. Nothing opens a Parquet file. That is the hard
rule, and it is what makes the point-in-time guarantee enforceable instead of
aspirational.

`dal` returns a lazy `DuckDBPyRelation`. Materialise at the edge only, with
`.pl()` for a polars frame or `.fetchall()` for Python values.

Run the whole notebook, or read it from the top. Each section is independent.

In [ ]:
import datetime as dt

import polars as pl

from sdp import dal

pl.Config.set_tbl_rows(15)
pl.Config.set_fmt_str_lengths(60)

con = dal.con()   # The one connection that owns every relation.
print(dal.status())

---
## 1. What is published

A partition exists only after its audit passed. A missing date therefore means
"no data" and never "bad data".

In [ ]:
for name, ds in dal.DATASETS.items():
    parts = dal.partitions(ds)
    span = f"{parts[0]} to {parts[-1]}" if parts else "nothing published"
    print(f"{name:24s} key={ds.key:10s} {len(parts):>4} partitions   {span}")

In [ ]:
# gaps() lists the XNYS sessions inside a range that have no partition.
# An empty list means the range is complete.
first, last = dal.coverage(dal.DAY_AGGS)
print("day aggregates", first, "to", last)
print("interior gaps:", dal.gaps(dal.DAY_AGGS, first, last) or "none")

---
## 2. The point-in-time guarantee

Splits and dividends hold **current state**. The endpoint has no `as_of`
parameter, so it gives the belief of the vendor *right now* about all of history.
The partition key is therefore the pull date.

Three accessors exist, and their names carry the semantics:

| Function | Meaning |
|---|---|
| `snapshot(ds, as_of)` | The newest pull at or before `as_of`. **Point-in-time.** |
| `snapshot_latest(ds)` | The most recent pull. Not point-in-time. |
| `history(ds)` | Every pull, stacked. For restatement work. |

In [ ]:
pulls = dal.partitions(dal.SPLITS)
print("pulls of massive_splits:", pulls)

# A read as of a date between the two pulls returns the older one, never the newer.
between = pulls[0] + dt.timedelta(days=3)
# Do not name this one `asof`. ASOF is a reserved word in DuckDB, so a query
# that reads `from asof` is a parser error.
snap = dal.snapshot(dal.SPLITS, between)
print(f"as of {between} the snapshot has",
      con.sql("select count(*) from snap").fetchone()[0], "rows")

In [ ]:
# The same read with no as_of would be lookahead, so dal makes you say which you
# want. Both of these are explicit about not being point-in-time.
latest = dal.snapshot_latest(dal.SPLITS)
print("latest pull rows:", con.sql("select count(*) from latest").fetchone()[0])

### The guard rails

Asking for a snapshot older than the first pull raises. That is the honest
answer. The endpoint has no `as_of` parameter, so the first pull is the oldest
snapshot that can ever exist.

In [ ]:
try:
    dal.snapshot(dal.SPLITS, dt.date(2020, 1, 1))
except dal.MissingPartition as exc:
    print("MissingPartition:", exc)

In [ ]:
# The two kinds of dataset cannot be confused. Each function refuses the wrong one.
for call, label in [
    (lambda: dal.series(dal.SPLITS), "series() on a current-state dataset"),
    (lambda: dal.snapshot(dal.DAY_AGGS, dt.date(2026, 8, 21)), "snapshot() on an event stream"),
]:
    try:
        call()
    except ValueError as exc:
        print(f"{label}\n    -> {exc}\n")

---
## 3. Restatement, and why the pull date is the partition key

This is the section that pays for the whole design. Two pulls exist, 13 days
apart. Compare them.

In [ ]:
h = dal.history(dal.SPLITS)
con.sql("select pull_date, count(*) as rows, count(distinct ticker) as tickers "
        "from h group by pull_date order by pull_date").pl()

The row count moved. Now find what changed. The obvious key is the vendor `id`.

In [ ]:
# Compare the first pull against the most recent one. Do not unpack:
# the number of pulls grows by one every day.
pulls = dal.partitions(dal.SPLITS)
old_pull, new_pull = pulls[0], pulls[-1]

con.sql(f'''
    select
        (select count(*) from h a
          where a.pull_date = date '{old_pull}'
            and not exists (select 1 from h b
                            where b.pull_date = date '{new_pull}' and b.id = a.id)
        ) as ids_only_in_the_old_pull,
        (select count(*) from h b
          where b.pull_date = date '{new_pull}'
            and not exists (select 1 from h a
                            where a.pull_date = date '{old_pull}' and a.id = b.id)
        ) as ids_only_in_the_new_pull
''').pl()

That reads as though hundreds of corporate actions disappeared. They did not.

Check whether the same event is still there under a different `id`, by matching
on `(ticker, execution_date)` instead.

In [ ]:
con.sql(f'''
    with dropped as (
        select a.* from h a
        where a.pull_date = date '{old_pull}'
          and not exists (select 1 from h b
                          where b.pull_date = date '{new_pull}' and b.id = a.id)
    ),
    new_pull as (select * from h where pull_date = date '{new_pull}')
    select
        count(*) as ids_that_vanished,
        count(*) filter (
            exists (select 1 from new_pull n
                    where n.ticker = d.ticker and n.execution_date = d.execution_date)
        ) as same_event_still_present_under_a_new_id
    from dropped d
''').pl()

**Every one of them is still present under a different id.** The vendor `id` is
not stable across pulls.

Two consequences:

1. A diff on `id` overstates the churn. It reports hundreds of deletions and
   insertions where the events did not change at all.
2. The dbt SCD Type 2 snapshot must not key on `id`. It would record a deletion
   and an insertion for every event whose id churned, on every pull. Key on
   `(ticker, execution_date)` instead, or on the identifier that decision 0007
   settles on.

Now measure the change that is real.

In [ ]:
# Restrict to the keys that are unambiguous in both pulls. Joining on
# (ticker, execution_date) without this fans out across the 209 duplicated
# split keys and overstates the count. Section 9 has the detail.
con.sql(f'''
    with a as (select * from h where pull_date = date '{old_pull}'),
         b as (select * from h where pull_date = date '{new_pull}'),
         ua as (select ticker, execution_date from a group by 1,2 having count(*) = 1),
         ub as (select ticker, execution_date from b group by 1,2 having count(*) = 1),
         k  as (select * from ua intersect select * from ub)
    select
        (select count(*)
           from a join b using (ticker, execution_date)
                  join k using (ticker, execution_date)
          where a.historical_adjustment_factor
                is distinct from b.historical_adjustment_factor) as factor_restated,
        (select count(*) from b
          where not exists (select 1 from a
                            where a.ticker = b.ticker
                              and a.execution_date = b.execution_date)) as genuinely_new_events
''').pl()

A restated adjustment factor is not cosmetic. It changes every adjusted price
before that event. A backtest that read the newer snapshot to price a trade dated
before the newer pull would have used information that did not exist at the time.

The `pull_date` partitions are what make this measurable. There is no
`updated_since` field on the endpoint, so a diff of complete snapshots is the only
mechanism available.

In [ ]:
# The tickers that were restated. These are the names to look at first.
con.sql(f'''
    with a as (select * from h where pull_date = date '{old_pull}'),
         b as (select * from h where pull_date = date '{new_pull}'),
         ua as (select ticker, execution_date from a group by 1,2 having count(*) = 1),
         ub as (select ticker, execution_date from b group by 1,2 having count(*) = 1),
         k  as (select * from ua intersect select * from ub)
    select a.ticker, a.execution_date, a.adjustment_type,
           a.historical_adjustment_factor as factor_before,
           b.historical_adjustment_factor as factor_after
    from a join b using (ticker, execution_date)
            join k using (ticker, execution_date)
    where a.historical_adjustment_factor
          is distinct from b.historical_adjustment_factor
    order by a.execution_date desc
    limit 15
''').pl()

---
## 4. Day aggregates

Unadjusted prices, one row for each ticker and session. Unadjusted is the point.
An adjusted price restates retroactively, and that would break the immutability of
a published partition.

In [ ]:
bars = dal.day_aggs()
con.sql("select count(*) as rows, count(distinct ticker) as tickers, "
        "min(date) as first_session, max(date) as last_session from bars").pl()

In [ ]:
con.sql('''
    select date, count(*) as tickers,
           round(sum(volume * close) / 1e9, 1) as dollar_volume_bn
    from bars group by date order by date
''').pl()

A detail worth knowing before you trust a type: `volume` is a floating point
number and not an integer. Most rows are fractional, because the consolidated tape
carries fractional share quantities.

In [ ]:
con.sql('''
    select count(*) as rows,
           count(*) filter (volume <> floor(volume)) as fractional_volume,
           round(100.0 * count(*) filter (volume <> floor(volume)) / count(*), 1) as pct
    from bars
''').pl()

---
## 5. The universe, and the identifier that is still an open question

Two independent filters build the universe, and both are recomputed for each date.
The instrument filter is the simple one.

In [ ]:
names = dal.tickers_on(dal.partitions(dal.TICKERS)[-1])
con.sql('''
    select type, count(*) as n
    from names group by type order by n desc limit 12
''').pl()

In [ ]:
con.sql('''
    select
        count(*) as all_instruments,
        count(*) filter (type = 'CS') as common_stock,
        count(*) filter (type = 'CS'
                         and primary_exchange in ('XNYS','XNAS','XASE')) as after_exchange_filter
    from names
''').pl()

### The identifier problem

A ticker symbol changes, and worse, it is reused. A symbol that a delisting frees
can go to a different company. A key on ticker would join the returns of two
companies into one series, and the discontinuity looks like a fat tail.

The intent was to key on `composite_figi`. Coverage blocks it. See
`docs/decisions/0007-identifier-strategy.md`.

In [ ]:
con.sql('''
    select
        count(*) as cs_rows,
        count(*) filter (composite_figi is null) as null_composite_figi,
        count(*) filter (share_class_figi is null) as null_share_class_figi,
        count(*) filter (cik is null) as null_cik
    from names where type = 'CS'
''').pl()

In [ ]:
# The landscape is the inverse of what you would expect. FIGI is complete for
# ETFs and patchy for common stock. CIK is the other way round.
con.sql('''
    select type, count(*) as n,
           count(*) filter (composite_figi is null) as null_figi,
           count(*) filter (cik is null) as null_cik
    from names
    where type in ('CS', 'ETF', 'ADRC', 'WARRANT', 'PFD')
    group by type order by n desc
''').pl()

The number that decides this question is the exposure **after** the liquidity
filter, not before it. If the null-FIGI names are a tail of recent listings and
shells, the liquidity filter removes them and the question does not matter. That
diagnostic needs the backfill.

---
## 6. Adjustment for corporate actions

The `historical_adjustment_factor` of the vendor is **cumulative**. It already
contains every later action. The adjustment is therefore one as-of lookup and not
a chained product.

> For a price on date D, find the first event whose `execution_date` is **after**
> D. Multiply the price by the factor of that event.

Check the semantics against a known value. AAPL split 2-for-1 in 2005, 7-for-1 in
2014 and 4-for-1 in 2020.

In [ ]:
splits = dal.splits(as_of=dal.partitions(dal.SPLITS)[-1])
con.sql('''
    select execution_date, adjustment_type,
           split_from, split_to,
           split_to / split_from as ratio,
           historical_adjustment_factor
    from splits where ticker = 'AAPL' order by execution_date
''').pl()

In [ ]:
# The 2005 factor must equal 1/2 * 1/7 * 1/4 = 1/56, compounded with the later
# two splits. Reproducing it from the ratios confirms the semantics.
expected = 1 / (2 * 7 * 4)
print(f"1 / (2 * 7 * 4) = {expected:.6f}")

actual = con.sql(
    "select historical_adjustment_factor from splits "
    "where ticker = 'AAPL' and execution_date = date '2005-02-28'"
).fetchone()[0]
print(f"vendor factor for 2005-02-28 = {actual}")
print("match:", round(expected, 6) == round(actual, 6))

### The boundary is strict

Split adjustment applies overnight. On the execution date all trading is already
adjusted, including the pre-market session. The join must be `> D` and not `>= D`.
An error of one day makes one very large false return for each split and each
name.

In [ ]:
# The as-of join, written out. This is the shape that the dbt staging model needs.
con.sql('''
    with universe as (
        select ticker, date, close from bars where ticker = 'AAPL'
    )
    select u.date, u.close,
           (select s.historical_adjustment_factor
              from splits s
             where s.ticker = u.ticker
               and s.execution_date > u.date      -- strictly after
             order by s.execution_date
             limit 1) as split_factor
    from universe u
    order by u.date
    limit 10
''').pl()

A null factor means that no split follows that date, so the price needs no split
adjustment. AAPL has no split after 2020, so recent bars are already on today's
share basis.

---
## 7. The audit invariants, checked against the live data

These are the properties that the audits enforce at ingest. Confirm that they hold
on what is published.

In [ ]:
# Splits classification. Every forward_split has a ratio above 1, every
# reverse_split below 1, and every stock_dividend above 1.
con.sql('''
    select adjustment_type,
           count(*) as n,
           min(split_to / split_from) as min_ratio,
           max(split_to / split_from) as max_ratio
    from splits group by adjustment_type order by n desc
''').pl()

In [ ]:
# Reverse splits outnumber forward splits by about two to one. Most are
# distressed microcaps doing a 1-for-10 to keep a listing.
con.sql('''
    select adjustment_type, count(*) as n,
           round(100.0 * count(*) / sum(count(*)) over (), 1) as pct
    from splits group by adjustment_type order by n desc
''').pl()

In [ ]:
# RYCEF. A Rolls-Royce ADR with a 1:72 stock dividend and a factor of 0.0.
# Rolls-Royce issued C Shares instead of a cash dividend. Those shares are not
# fungible with the ordinary shares, so no valid price adjustment exists and the
# vendor emits 0.0 rather than 1/72. Treating it as a split would manufacture a
# one-day fall of 98.6%.
con.sql('''
    select ticker, execution_date, adjustment_type, split_from, split_to,
           historical_adjustment_factor
    from splits
    where historical_adjustment_factor <= 0
    order by execution_date desc
    limit 10
''').pl()

In [ ]:
# The dividend factor is different. A null there is structural and not a defect.
# It means the vendor had no price on the ex-date to compute (1 - D/P) against.
divs = dal.dividends(as_of=dal.partitions(dal.DIVIDENDS)[-1])
con.sql('''
    select count(*) as rows,
           count(*) filter (historical_adjustment_factor is null) as null_factor,
           round(100.0 * count(*) filter (historical_adjustment_factor is null)
                 / count(*), 1) as pct_null,
           count(*) filter (currency is not null and currency <> 'USD') as not_usd
    from divs
''').pl()

In [ ]:
# Restrict to the tickers that actually trade on the ingested tape. The null rate
# collapses. The nulls are dividends on securities that never trade there:
# foreign issuers, OTC names that do not report, and fund share classes.
con.sql('''
    with traded as (select distinct ticker from bars)
    select
        case when d.ticker in (select ticker from traded)
             then 'in the day aggregates' else 'not in the day aggregates' end as group_,
        count(*) as rows,
        round(100.0 * count(*) filter (d.historical_adjustment_factor is null)
              / count(*), 1) as pct_null_factor
    from divs d group by 1
''').pl()

That second number is now settled. Before the backfill the bars covered a few
weeks, so any name that delisted earlier was counted as "not traded", which is
exactly the survivorship-sensitive set. With five years of bars the split is
clean: **0.97%** null inside the traded set against **39.08%** outside it. The
nulls are dividends on securities that never trade on this tape.

So `adj_close_total` is the default column and not the fallback. That was the
open caveat in `CLAUDE.md` and it is closed.

---
## 8. The universe, now that the backfill has run

The lake holds 1,255 sessions from 2021-08-23. Everything below is the first
look at the models on real history rather than on three weeks.

In [ ]:
import duckdb

from sdp.config import settings

# The warehouse is dbt's output. Read it read-only, so a build can run beside
# this notebook.
wh = duckdb.connect(str(settings.warehouse_path), read_only=True)

wh.sql('''
    select date, count(*) filter (in_universe) as names
    from main_staging.stg_universe
    group by 1 order by 1
''').pl()

In [ ]:
# The size of the universe over time. The target was 1,000 to 2,000 names.
wh.sql('''
    with per_date as (
        select date, count(*) filter (in_universe) as n
        from main_staging.stg_universe group by 1
    )
    select date_trunc('year', date) as year,
           round(avg(n)) as avg_names,
           min(n) as min_names,
           max(n) as max_names
    from per_date group by 1 order by 1
''').pl()

Two things to read here.

**The universe is about 2,960 names at the median, which is half again above the
1,000 to 2,000 target.** The thresholds are dbt vars, so this is a dial and not a
defect: `min_price` is $5 and `min_dollar_volume` is $1M. Tightening the ADV floor
is the first thing to try, and varying it is a robustness check rather than
tidying, because a signal that only works at the loose end is an illiquidity
premium and not alpha.

**The first 59 sessions have no universe at all.** A name needs 60 sessions of
history to pass `min_days_since_first_bar`, and that history does not exist for
the start of the lake. The usable window therefore begins on 2021-11-15, giving
1,196 sessions and not 1,255. Say that in a writeup rather than quietly starting
the sample there.

In [ ]:
# Where the names are lost. Each filter is a column, so an empty universe is
# diagnosable instead of mysterious.
wh.sql('''
    select
        count(*)                                  as rows,
        count(*) filter (passes_instrument)       as after_instrument,
        count(*) filter (passes_instrument and passes_price)      as and_price,
        count(*) filter (passes_instrument and passes_price
                         and passes_adv)          as and_adv,
        count(*) filter (in_universe)             as in_universe
    from main_staging.stg_universe
    where date = (select max(date) from main_staging.stg_universe)
''').pl()

---
## 9. The open questions, answered

`python -m sdp.diagnostics` runs these against the whole lake. The queries are
repeated here so the numbers can be read next to the reasoning.

### The identifier (decision 0007)

Ticker reuse is real, and the size of it decides whether a simple key is
defensible.

In [ ]:
names = dal.tickers()

con.sql('''
    with per_ticker as (
        select ticker, count(distinct composite_figi) as figis
        from names where type = 'CS' and composite_figi is not null
        group by 1
    )
    select count(*) as cs_tickers,
           count(*) filter (figis > 1) as more_than_one_figi,
           count(*) filter (figis > 2) as more_than_two
    from per_ticker
''').pl()

886 of 9,018 common stock tickers carry more than one FIGI across the window, and
67 carry more than two. That is one ticker in ten, so keying on the symbol is not
defensible. It would splice two companies into one return series and the
discontinuity would read as a fat tail.

The number that decides the fallback rule is the FIGI gap **after** the liquidity
screen, not before it.

In [ ]:
wh.sql('''
    select
        count(*) as cs_rows,
        round(100.0 * count(*) filter (security_key is null
                                       or key_rule <> 'share_class_figi')
              / count(*), 2) as pct_not_on_figi,
        count(*) filter (in_universe) as after_the_screen,
        round(100.0 * count(*) filter (in_universe
                                       and key_rule <> 'share_class_figi')
              / nullif(count(*) filter (in_universe), 0), 2) as pct_not_on_figi_in_universe
    from main_staging.stg_universe
    where type = 'CS'
''').pl()

The screen helps and it does not rescue the situation: the gap falls from 15.4%
to **9.9%**, which is still one row in ten falling back to CIK or to the ticker.
Only 2,053 rows have neither a FIGI nor a CIK.

So the coalesce with a recorded `key_rule` is the right answer, and the fallback
rate is large enough that a study should report results with and without the
fallback rows.

### The ambiguous event key (decision 0010)

In [ ]:
splits = dal.splits(as_of=dal.partitions(dal.SPLITS)[-1])

con.sql('''
    with per_key as (
        select ticker, execution_date,
               count(*) as rows,
               count(distinct historical_adjustment_factor) as factors
        from splits group by 1, 2
    )
    select count(*) as distinct_keys,
           count(*) filter (rows > 1) as duplicated,
           count(*) filter (rows > 1 and factors > 1) as and_disagreeing
    from per_key
''').pl()

209 split keys are duplicated and **every one of them disagrees about the
factor**. 33 fall inside the price window. Dividends are worse: 13,864 duplicated
keys of which 7,449 disagree, and 3,773 land in the window.

This is not a curiosity. An as-of join against the raw rows fans out, so one
price row becomes two with two different adjusted prices. `stg_corporate_actions`
resolves the key before the join, which is why the adjustment model joins against
it and never against the lake.

The next cell shows the failure, on purpose.

In [ ]:
# The trap, demonstrated. Joining prices to the raw splits on the event key
# multiplies rows wherever the key is duplicated.
con.sql('''
    with one_name as (
        select ticker, execution_date
        from splits group by 1, 2 having count(*) > 1 limit 1
    )
    select s.ticker, s.execution_date, s.split_from, s.split_to,
           s.historical_adjustment_factor
    from splits s join one_name using (ticker, execution_date)
''').pl()

---
## 10. Restatement, measured over two pulls

`python -m sdp.restatement` compares the earliest corporate action pull against
the latest. Decision 0011 accepts that a study of the historical window cannot be
point-in-time on corporate actions, and requires the residual lookahead to be a
measurement rather than an assumption. This is that measurement.

In [ ]:
h = dal.history(dal.SPLITS)
old_pull, new_pull = dal.partitions(dal.SPLITS)[0], dal.partitions(dal.SPLITS)[-1]
print(f"{old_pull} against {new_pull}")

con.sql(f'''
    with a as (select * from h where pull_date = date '{old_pull}'),
         b as (select * from h where pull_date = date '{new_pull}')
    select
        (select count(*) from a) as rows_before,
        (select count(*) from b) as rows_after,
        (select count(*) from a where not exists
            (select 1 from b where b.id = a.id)) as ids_gone,
        (select count(*) from b where not exists
            (select 1 from a where a.id = b.id)) as ids_new
''').pl()

Read on the vendor `id`, hundreds of events appear and disappear in two weeks.
Almost none of that is real. The `id` is not stable across pulls, so the same
event returns under a new one. Match on the event instead.

In [ ]:
con.sql(f'''
    with a as (select * from h where pull_date = date '{old_pull}'),
         b as (select * from h where pull_date = date '{new_pull}')
    select
        (select count(*) from a where not exists
            (select 1 from b where b.ticker = a.ticker
                               and b.execution_date = a.execution_date)) as events_gone,
        (select count(*) from b where not exists
            (select 1 from a where a.ticker = b.ticker
                               and a.execution_date = b.execution_date)) as events_new
''').pl()

5 gone and 82 new, against 400 and 479 on the id. The rest was churn.

### How much of the restatement is mechanical

The adjustment factor is cumulative, so it embeds every later action. When a
ticker has a new dividend, **every earlier factor for that ticker changes by
design**. That is not the vendor correcting itself, and the two must be separated
before the lookahead can be called a risk.

The join below keeps only the keys that are unambiguous in both pulls. Without
that restriction it fans out across the 13,926 duplicated dividend keys and
overstates the count by about a quarter. That mistake was made while writing this
notebook, which is the argument for section 9 in one line.

In [ ]:
hd = dal.history(dal.DIVIDENDS)
d_old, d_new = dal.partitions(dal.DIVIDENDS)[0], dal.partitions(dal.DIVIDENDS)[-1]

con.sql(f'''
    with a as (select * from hd where pull_date = date '{d_old}'),
         b as (select * from hd where pull_date = date '{d_new}'),
         ua as (select ticker, ex_dividend_date from a group by 1,2 having count(*) = 1),
         ub as (select ticker, ex_dividend_date from b group by 1,2 having count(*) = 1),
         k  as (select * from ua intersect select * from ub),
         changed as (
             select a.ticker
             from a join b using (ticker, ex_dividend_date)
                     join k using (ticker, ex_dividend_date)
             where a.historical_adjustment_factor
                   is distinct from b.historical_adjustment_factor
         ),
         went_ex as (
             select distinct ticker from b
             where ex_dividend_date > date '{d_old}'
               and ex_dividend_date <= date '{d_new}'
         )
    select count(*) as restated,
           count(*) filter (ticker in (select ticker from went_ex)) as mechanical,
           count(*) filter (ticker not in (select ticker from went_ex)) as vendor_revision
    from changed
''').pl()

**98.1% of it is mechanical.** 87,267 of 88,972 restated factors sit on a ticker
that went ex inside the window, which is the cumulative factor doing exactly what
decision 0005 describes. Only 1,705 rows are the vendor changing its mind about
an event that did not move.

That is a much better result than the raw count suggests, and it is the number
that belongs in a writeup. The lookahead from reading a later corporate action
snapshot is dominated by a mechanism that is understood and predictable, not by
arbitrary revision.

---
## 11. A typical workflow

The loop from here on. Nothing below builds a signal worth trading. The point is
the shape of the loop, and finding the plumbing errors on a result nobody is
attached to.

**1. Pull the universe for a date range, from the model and never by hand.**

In [ ]:
panel = wh.sql('''
    select u.date, u.security_key, u.ticker, u.close, u.adv,
           p.adj_close_total
    from main_staging.stg_universe u
    join main_staging.stg_prices_adjusted p
      on p.date = u.date and p.ticker = u.ticker
    where u.in_universe
      and u.date >= date '2024-01-01'
''')
print(f"{panel.count('*').fetchone()[0]:,} name-days")
panel.limit(5).pl()

**2. Build a signal. This one is a deliberately weak five-day reversal, with no
residualization.**

It is chosen because it is easy to get right and easy to check. A plumbing error
shows up as an implausibly good result, and on a signal this crude an
implausibly good result is obviously wrong.

In [ ]:
signal = wh.sql('''
    with px as (
        select u.date, u.security_key, p.adj_close_total as px
        from main_staging.stg_universe u
        join main_staging.stg_prices_adjusted p
          on p.date = u.date and p.ticker = u.ticker
        where u.in_universe and p.adj_close_total is not null
    ),
    with_lags as (
        select date, security_key, px,
               lag(px, 5) over w as px_5,
               lead(px, 1) over w as px_next
        from px
        window w as (partition by security_key order by date)
    )
    select date, security_key,
           -- The signal. Negative of the trailing five-day return.
           -(px / px_5 - 1)          as reversal,
           -- The thing it must predict. The next day return.
           px_next / px - 1          as fwd_return
    from with_lags
    where px_5 is not null and px_next is not null
''')
print(f"{signal.count('*').fetchone()[0]:,} rows with a signal and a forward return")

Two things in that query are the whole point.

`lead(px, 1)` is the forward return, and it is the only place the future is
allowed to appear. If a signal column ever reads `lead`, the result is
meaningless and it will look excellent.

The partition is `security_key` and not `ticker`, so a symbol that changed hands
does not have two companies' prices lagged into one series.

**3. Score it. The information coefficient is the cross-sectional rank
correlation between the signal and the forward return, computed for each date and
then averaged.**

In [ ]:
# `signal` was built on `wh`, and a relation belongs to the connection that
# made it. Querying it from `con` raises.
ic = wh.sql('''
    with daily as (
        select date, corr(rs, rf) as ic
        from (
            select date,
                   rank() over (partition by date order by reversal)   as rs,
                   rank() over (partition by date order by fwd_return) as rf
            from signal
        )
        group by date
        having count(*) > 100
    )
    select count(*)                      as days,
           round(avg(ic), 5)             as mean_ic,
           round(stddev(ic), 5)          as sd_ic,
           round(avg(ic) / (stddev(ic) / sqrt(count(*))), 2) as t_stat
    from daily
''')
ic.pl()

Whatever that number is, treat it as a plumbing check and not as a result. The
things that would make it a result are all still missing:

- **No residualization.** A raw reversal is mostly a bet against whatever the
  market and the sectors did. Decision 0012 wants the PCA risk model first, and
  the signal defined on the residual.
- **No purging.** Overlapping windows leak across a naive train and test split.
- **No costs.** Reversal turns over daily and it is the strategy most easily
  destroyed by the spread. A backtest without costs is not evidence.
- **No bid-ask bounce control.** This is the one that matters most here.
  Consecutive closes in a wide-spread name alternate between the bid and the ask,
  which manufactures negative serial correlation out of nothing and cannot be
  harvested, because capturing it means crossing the spread that created it. The
  liquidity screen exists for this reason, and the honest check is whether the IC
  survives a much tighter ADV floor.

**4. Vary the thresholds, because that is the robustness check.**

Rebuild with a tighter floor and score it again:

```bash
python -m sdp.transform build --vars '{min_dollar_volume: 10000000}'
```

If the IC is 0.04 at a $1M floor and 0.005 at $10M, the signal is an illiquidity
premium that cannot be harvested. That comparison is the result, not the first
number.

**5. Write the kill-log entry whatever happens.** Decision 0012 defines what a
finished study contains.

---
## 12. Where things stand

Built and measured:

- 1,255 sessions of bars and of tickers from 2021-08-23, 638 of short volume, 119
  short interest settlements. No gaps.
- Four staging models and 31 dbt nodes passing on the real history.
- A usable universe of about 2,960 names a day from 2021-11-15.

Open, with the numbers now in hand:

- **The universe is half again above target.** Tighten `min_dollar_volume` first.
- **The identifier fallback is 9.9% after the screen.** Decision 0007 can be
  closed on the coalesce rule, and a study should report with and without the
  fallback rows.
- **The strategic choice.** US cross-sectional daily stat-arb is crowded. Short
  interest as a crowding signal has the full window that the prices allow, and it
  is the study that almost nobody else can write.

Next: the evaluation harness, then the PCA risk model, then residual reversal.